# 🌍 Lahore AQI Forecasting — Model Training
## Spatio-Temporal Air Quality Prediction via Hybrid ConvLSTM

**Pipeline:** Baselines → BiLSTM → ConvLSTM → Model Comparison

**Runtime:** Use **GPU** (Runtime → Change runtime type → T4 GPU)

## 1. Setup & Data Upload

In [ ]:
# Mount Google Drive (upload your project here)
from google.colab import drive
drive.mount('/content/drive')

# Set project root
import os
PROJECT_ROOT = '/content/drive/MyDrive/Air-Quality-Forecasting'  # <-- CHANGE THIS to your Drive path
os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# OR: Upload sequence files directly (if not using Drive)
# from google.colab import files
# uploaded = files.upload()  # Upload lstm_X_train.npy, lstm_y_train.npy, etc.

In [ ]:
!pip install -q xgboost scikit-learn joblib

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

# Check GPU
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
for gpu in gpus:
    print(f'  {gpu}')

## 2. Load Sequences

In [ ]:
SEQ_DIR = os.path.join(PROJECT_ROOT, 'data', 'sequences')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'saved_models')
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, 'outputs')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Load LSTM sequences
print('Loading LSTM sequences...')
X_train = np.load(os.path.join(SEQ_DIR, 'lstm_X_train.npy'))
y_train = np.load(os.path.join(SEQ_DIR, 'lstm_y_train.npy'))
X_test = np.load(os.path.join(SEQ_DIR, 'lstm_X_test.npy'))
y_test = np.load(os.path.join(SEQ_DIR, 'lstm_y_test.npy'))

print(f'X_train: {X_train.shape} ({X_train.nbytes / 1e6:.0f} MB)')
print(f'y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape} ({X_test.nbytes / 1e6:.0f} MB)')
print(f'y_test:  {y_test.shape}')
print(f'\ny_train stats: mean={y_train.mean():.2f}, std={y_train.std():.2f}, min={y_train.min():.2f}, max={y_train.max():.2f}')

In [ ]:
# Load ConvLSTM sequences
print('Loading ConvLSTM sequences...')
conv_X_train = np.load(os.path.join(SEQ_DIR, 'convlstm_X_train.npy'))
conv_y_train = np.load(os.path.join(SEQ_DIR, 'convlstm_y_train.npy'))
conv_X_test = np.load(os.path.join(SEQ_DIR, 'convlstm_X_test.npy'))
conv_y_test = np.load(os.path.join(SEQ_DIR, 'convlstm_y_test.npy'))

print(f'ConvLSTM X_train: {conv_X_train.shape} ({conv_X_train.nbytes / 1e9:.2f} GB)')
print(f'ConvLSTM y_train: {conv_y_train.shape}')
print(f'ConvLSTM X_test:  {conv_X_test.shape} ({conv_X_test.nbytes / 1e9:.2f} GB)')
print(f'ConvLSTM y_test:  {conv_y_test.shape}')

## 3. Evaluation Framework

In [ ]:
class ModelEvaluator:
    """Computes and stores metrics for all models."""

    def __init__(self):
        self.results = []

    def evaluate(self, y_true, y_pred, model_name):
        y_t = y_true.flatten()
        y_p = y_pred.flatten()
        mask = np.isfinite(y_t) & np.isfinite(y_p)
        y_t, y_p = y_t[mask], y_p[mask]

        mae = mean_absolute_error(y_t, y_p)
        rmse = np.sqrt(mean_squared_error(y_t, y_p))
        r2 = r2_score(y_t, y_p)

        nonzero = y_t != 0
        mape = mean_absolute_percentage_error(y_t[nonzero], y_p[nonzero]) * 100 if nonzero.sum() > 0 else float('nan')

        within_10 = (np.abs(y_t - y_p) <= 10).mean() * 100
        within_25pct = (np.abs(y_t - y_p) / np.maximum(y_t, 1e-8) <= 0.25).mean() * 100

        metrics = {
            'Model': model_name,
            'MAE': round(mae, 4),
            'RMSE': round(rmse, 4),
            'R²': round(r2, 4),
            'MAPE%': round(mape, 2),
            'Within±10': round(within_10, 2),
            'Within±25%': round(within_25pct, 2),
        }
        self.results.append(metrics)
        print(f'  [{model_name}] MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, MAPE={mape:.2f}%')
        return metrics

    def comparison_table(self):
        df = pd.DataFrame(self.results).sort_values('RMSE')
        return df

evaluator = ModelEvaluator()
print('Evaluator ready.')

## 4. Baseline Models

In [ ]:
print('=' * 60)
print('  BASELINE MODELS')
print('=' * 60)

# 1. Naive Persistence: predict PM2.5(t+6h) = PM2.5(t)
pred_naive = X_test[:, -1, 0]  # last timestep, first feature (pm25)
evaluator.evaluate(y_test, pred_naive, 'Naive Persistence')

# 2. Historical Mean
pred_mean = np.full(len(y_test), y_train.mean())
evaluator.evaluate(y_test, pred_mean, 'Historical Mean')

In [ ]:
# 3. Ridge Regression
print('\nTraining Ridge Regression...')
X_flat_train = X_train.reshape(X_train.shape[0], -1)
X_flat_test = X_test.reshape(X_test.shape[0], -1)

t0 = time.time()
ridge = Ridge(alpha=1.0)
ridge.fit(X_flat_train, y_train)
pred_ridge = ridge.predict(X_flat_test)
print(f'  Time: {time.time()-t0:.1f}s')
evaluator.evaluate(y_test, pred_ridge, 'Ridge Regression')

joblib.dump(ridge, os.path.join(MODELS_DIR, 'ridge_baseline.pkl'))

In [ ]:
# 4. Random Forest
print('\nTraining Random Forest...')
t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    max_samples=0.5,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_flat_train, y_train)
pred_rf = rf.predict(X_flat_test)
print(f'  Time: {time.time()-t0:.1f}s')
evaluator.evaluate(y_test, pred_rf, 'Random Forest')

joblib.dump(rf, os.path.join(MODELS_DIR, 'rf_baseline.pkl'))

In [ ]:
# 5. XGBoost
print('\nTraining XGBoost...')
from xgboost import XGBRegressor

t0 = time.time()
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    tree_method='hist',
    device='cuda',  # GPU-accelerated XGBoost on Colab
    random_state=42,
    verbosity=0,
)
xgb.fit(X_flat_train, y_train)
pred_xgb = xgb.predict(X_flat_test)
print(f'  Time: {time.time()-t0:.1f}s')
evaluator.evaluate(y_test, pred_xgb, 'XGBoost')

joblib.dump(xgb, os.path.join(MODELS_DIR, 'xgb_baseline.pkl'))

In [ ]:
print('\n\nBaseline Results:')
display(evaluator.comparison_table())

## 5. BiLSTM Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

def build_lstm(input_shape):
    model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True, input_shape=input_shape), name='bilstm_1'),
        Dropout(0.3),
        Bidirectional(LSTM(64, return_sequences=False), name='bilstm_2'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='linear'),
    ], name='BiLSTM_Forecaster')

    model.compile(optimizer=Adam(1e-3), loss='huber', metrics=['mae'])
    return model

lstm_model = build_lstm((X_train.shape[1], X_train.shape[2]))
lstm_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint(os.path.join(MODELS_DIR, 'lstm_best.keras'), monitor='val_loss', save_best_only=True),
]

print('Training BiLSTM...')
t0 = time.time()
lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=256,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1,
)
print(f'\nTraining time: {(time.time()-t0)/60:.1f} min')

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(lstm_history.history['loss'], label='Train Loss')
ax1.plot(lstm_history.history['val_loss'], label='Val Loss')
ax1.set_title('BiLSTM Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Huber Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(lstm_history.history['mae'], label='Train MAE')
ax2.plot(lstm_history.history['val_mae'], label='Val MAE')
ax2.set_title('BiLSTM MAE Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'lstm_training_curves.png'), dpi=150)
plt.show()

In [ ]:
# Evaluate BiLSTM
pred_lstm = lstm_model.predict(X_test, batch_size=512, verbose=0).flatten()
evaluator.evaluate(y_test, pred_lstm, 'BiLSTM')

lstm_model.save(os.path.join(MODELS_DIR, 'lstm_final.keras'))
print('BiLSTM saved.')

## 6. ConvLSTM Model

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, ConvLSTM2D, BatchNormalization, Dropout, Conv2D, Reshape
)

def build_convlstm(input_shape):
    """input_shape: (seq_len, grid_H, grid_W, n_features)"""
    seq_len, grid_h, grid_w, n_features = input_shape

    inputs = Input(shape=input_shape, name='input_grid')

    x = ConvLSTM2D(64, (3,3), padding='same', return_sequences=True, activation='relu', name='convlstm_1')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = ConvLSTM2D(32, (3,3), padding='same', return_sequences=False, activation='relu', name='convlstm_2')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Conv2D(16, (3,3), padding='same', activation='relu')(x)
    outputs = Conv2D(1, (1,1), padding='same', activation='linear')(x)
    outputs = Reshape((grid_h, grid_w))(outputs)

    model = Model(inputs, outputs, name='ConvLSTM_Forecaster')
    model.compile(optimizer=Adam(1e-3), loss='huber', metrics=['mae'])
    return model

convlstm_model = build_convlstm(conv_X_train.shape[1:])
convlstm_model.summary()

In [ ]:
conv_callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint(os.path.join(MODELS_DIR, 'convlstm_best.keras'), monitor='val_loss', save_best_only=True),
]

print('Training ConvLSTM...')
t0 = time.time()
conv_history = convlstm_model.fit(
    conv_X_train, conv_y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.15,
    callbacks=conv_callbacks,
    verbose=1,
)
print(f'\nTraining time: {(time.time()-t0)/60:.1f} min')

In [ ]:
# Plot ConvLSTM training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(conv_history.history['loss'], label='Train Loss')
ax1.plot(conv_history.history['val_loss'], label='Val Loss')
ax1.set_title('ConvLSTM Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Huber Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(conv_history.history['mae'], label='Train MAE')
ax2.plot(conv_history.history['val_mae'], label='Val MAE')
ax2.set_title('ConvLSTM MAE Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'convlstm_training_curves.png'), dpi=150)
plt.show()

In [ ]:
# Evaluate ConvLSTM
pred_convlstm = convlstm_model.predict(conv_X_test, batch_size=8, verbose=0)
evaluator.evaluate(conv_y_test, pred_convlstm, 'ConvLSTM')

convlstm_model.save(os.path.join(MODELS_DIR, 'convlstm_final.keras'))
print('ConvLSTM saved.')

## 7. Model Comparison

In [ ]:
print('\n' + '=' * 80)
print('  FINAL MODEL COMPARISON (sorted by RMSE)')
print('=' * 80)
comparison = evaluator.comparison_table()
display(comparison)
comparison.to_csv(os.path.join(OUTPUTS_DIR, 'model_comparison.csv'), index=False)
print(f'\nSaved to {OUTPUTS_DIR}/model_comparison.csv')

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = ['#FF6B6B', '#FFA07A', '#FFD700', '#90EE90', '#87CEEB', '#DDA0DD', '#B0C4DE']

for idx, metric in enumerate(['MAE', 'RMSE', 'R²']):
    ax = axes[idx]
    models = comparison['Model'].values
    values = comparison[metric].values
    bars = ax.barh(models, values, color=colors[:len(models)])
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.set_xlabel(metric)

    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max(values)*0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'model_comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Prediction Visualization

In [ ]:
# Time-series comparison: actual vs predicted (best model)
best_model_name = comparison.iloc[0]['Model']

# Get predictions for best model
pred_map = {
    'Naive Persistence': pred_naive,
    'Historical Mean': pred_mean,
    'Ridge Regression': pred_ridge,
    'Random Forest': pred_rf,
    'XGBoost': pred_xgb,
    'BiLSTM': pred_lstm,
}

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Top: first 500 test samples
n_show = 500
ax = axes[0]
ax.plot(y_test[:n_show], label='Actual PM2.5', color='black', linewidth=1.5, alpha=0.8)
for name in ['Ridge Regression', 'XGBoost', 'BiLSTM']:
    if name in pred_map:
        ax.plot(pred_map[name][:n_show], label=name, alpha=0.7, linewidth=1)
ax.set_title(f'PM2.5 Forecast Comparison (first {n_show} test samples)', fontsize=14)
ax.set_xlabel('Time Step (hours)')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Bottom: scatter plot (actual vs predicted) for best model
ax = axes[1]
best_pred = pred_map.get(best_model_name, pred_lstm)
ax.scatter(y_test, best_pred, alpha=0.1, s=2, color='steelblue')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_title(f'{best_model_name}: Actual vs Predicted', fontsize=14)
ax.set_xlabel('Actual PM2.5 (µg/m³)')
ax.set_ylabel('Predicted PM2.5 (µg/m³)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'prediction_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Error distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, pred) in enumerate([('Ridge', pred_ridge), ('XGBoost', pred_xgb), ('BiLSTM', pred_lstm)]):
    errors = y_test - pred
    ax = axes[idx]
    ax.hist(errors, bins=80, alpha=0.7, color=['#FF6B6B', '#90EE90', '#87CEEB'][idx], edgecolor='black', linewidth=0.5)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_title(f'{name} Error Distribution', fontsize=12)
    ax.set_xlabel('Error (µg/m³)')
    ax.set_ylabel('Count')
    ax.text(0.02, 0.98, f'Mean error: {errors.mean():.2f}\nStd: {errors.std():.2f}',
            transform=ax.transAxes, va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'error_distributions.png'), dpi=150)
plt.show()

## 9. Download Results

In [ ]:
# List all saved files
print('\nSaved Models:')
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(os.path.join(MODELS_DIR, f)) / 1e6
    print(f'  {f}: {size:.1f} MB')

print('\nOutputs:')
for f in os.listdir(OUTPUTS_DIR):
    size = os.path.getsize(os.path.join(OUTPUTS_DIR, f)) / 1e6
    print(f'  {f}: {size:.1f} MB')

print('\n\u2705 All done! Models and outputs are saved in your Google Drive.')